In [3]:
import torch
input_ids=torch.tensor([2,3,4,5,1])
vocab_size=6
output_dim=3

embedding_layer=torch.nn.Embedding(vocab_size,output_dim)

In [4]:
print(embedding_layer.weight)

Parameter containing:
tensor([[-0.4750,  0.6896, -0.5333],
        [ 0.9504, -0.6981, -0.0574],
        [ 0.7341,  0.1681,  1.2342],
        [ 0.6839,  0.8263,  0.4100],
        [ 0.1146, -0.7548,  0.4846],
        [ 0.2029, -0.4736, -0.7346]], requires_grad=True)


In [7]:
print(embedding_layer(torch.tensor([2])))

tensor([[0.7341, 0.1681, 1.2342]], grad_fn=<EmbeddingBackward0>)


In [8]:
print(embedding_layer(input_ids))

tensor([[ 0.7341,  0.1681,  1.2342],
        [ 0.6839,  0.8263,  0.4100],
        [ 0.1146, -0.7548,  0.4846],
        [ 0.2029, -0.4736, -0.7346],
        [ 0.9504, -0.6981, -0.0574]], grad_fn=<EmbeddingBackward0>)


In [9]:
vocab_size=50527
output_dim=256
token_embedding_layer=torch.nn.Embedding(vocab_size,output_dim)

In [14]:
with open("wizard_of_oz.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:1000])

Total number of character: 252060
﻿The Project Gutenberg eBook of Dorothy and the Wizard in Oz
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

Title: Dorothy and the Wizard in Oz

Author: L. Frank Baum

Illustrator: John R. Neill


        
Release date: September 10, 2007 [eBook #22566]

Language: English

Other information and formats: www.gutenberg.org/ebooks/22566

Credits: Produced by Chris Curnow, Joseph Cooper, Janet Blenkinship
        and the Online Distributed Proofreading Team at
        http://www.pgdp.net


*** START OF THE PROJECT GUTENBERG EBOOK DOROTHY AND THE WIZARD IN OZ ***

In [15]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [18]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [19]:

import tiktoken
max_length = 6
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [20]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[  171,   119,   123,   464,  4935, 20336],
        [46566,   286, 40349,   290,   262, 16884],
        [  287, 18024,   198,   220,   220,   220],
        [  220,   198,  1212, 46566,   318,   329],
        [  262,   779,   286,  2687,  6609,   287],
        [  262,  1578,  1829,   290,   198,  1712],
        [  584,  3354,   286,   262,   995,   379],
        [  645,  1575,   290,   351,  2048,   645]])

Inputs shape:
 torch.Size([8, 6])


In [21]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 6, 256])


In [22]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [23]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

torch.Size([6, 256])


In [24]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 6, 256])
